<a href="https://colab.research.google.com/github/satyamkarn100-ctrl/FinDocAI/blob/main/02_LiLt_label_projection_and_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets accelerate seqeval huggingface_hub

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import json
import pandas as pd
from pathlib import Path
from huggingface_hub import snapshot_download

In [ ]:
REPO_ID = "Satyamkarn100/FinDoc-AI"
DATA_DIR = Path('/content/findoc_data')

snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=DATA_DIR,
    allow_patterns=["manifests/*", "ocr/doctr_default/*.jsonl"]
)

manifest = pd.read_csv(DATA_DIR / "manifests/manifest.csv")
print("Manifest shape:", manifest.shape)
print(manifest["split"].value_counts())



Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Manifest shape: (10000, 6)
split
train    6400
test     2000
val      1600
Name: count, dtype: int64


In [ ]:
ocr_data = {}
for shard_file in sorted((DATA_DIR / "ocr/doctr_default").glob("shard_*.jsonl")):
    with open(shard_file, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            ocr_data[rec["filename"]] = rec

print("\nTotal OCR Invoices Loaded from HF:", len(ocr_data))


Total OCR Invoices Loaded from HF: 10000


In [ ]:
import os, json, zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download

ANNOT_DIR = DATA_DIR / "Annotations"

# 1. Load ground-truth annotations (from HF if uploaded, otherwise extract only Annotations from Zenodo)
if not ANNOT_DIR.exists():
    try:
        zip_path = hf_hub_download(repo_id=REPO_ID, repo_type="dataset", filename="annotations.zip")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(ANNOT_DIR)
        print("Loaded annotations from Hugging Face.")
    except Exception:
        print("annotations.zip not found on HF. Extracting Annotations folder from Zenodo...")
        !wget -q -c "https://zenodo.org/records/10371464/files/FATURA2.zip?download=1" -O /content/FATURA2.zip
        !unzip -q /content/FATURA2.zip "invoices_dataset_final/Annotations/*" -d /content/FATURA_ANN
        ANNOT_DIR = Path("/content/FATURA_ANN/invoices_dataset_final/Annotations")
        print("Extracted annotations from Zenodo.")

print("Annotations subfolders:", os.listdir(ANNOT_DIR))

# 2. Inspect a sample OCR record
sample_name = "Template1_Instance0.jpg"
sample_ocr = ocr_data[sample_name]
print("\n--- OCR SAMPLE ---")
print("Keys:", list(sample_ocr.keys()), "| Size:", sample_ocr["width"], "x", sample_ocr["height"])
print("First 2 words:", sample_ocr["words"][:2])

# 3. Inspect the matching ground-truth JSON
json_matches = list(ANNOT_DIR.rglob("Template1_Instance0.json"))
print("\n--- FOUND JSON PATHS ---")
for p in json_matches:
    print(" ", p)

with open(json_matches[0], encoding="utf-8") as f:
    sample_ann = json.load(f)

print(f"\nTop keys ({json_matches[0].parent.name}):", list(sample_ann.keys()))
for k, v in list(sample_ann.items())[:5]:
    print(f"  {k}: {str(v)[:200]}")

annotations.zip not found on HF. Extracting Annotations folder from Zenodo...
Extracted annotations from Zenodo.
Annotations subfolders: ['.virtual_documents', 'layoutlm_HF_format', 'COCO_compatible_format', 'Original_Format']

--- OCR SAMPLE ---
Keys: ['filename', 'width', 'height', 'words'] | Size: 595 x 841
First 2 words: [['00', 0.8259, 0.0514, 0.03125, 0.09833, 0.06445, 0], ['Address:16424', 0.9464, 0.04726, 0.1709, 0.19357, 0.18457, 1]]

--- FOUND JSON PATHS ---
  /content/FATURA_ANN/invoices_dataset_final/Annotations/Original_Format/Template1_Instance0.json

Top keys (Original_Format): ['TABLE', 'INVOICE_INFO', 'BUYER', 'DATE', 'DISCOUNT', 'DUE_DATE', 'GSTIN_BUYER', 'NOTE', 'PAYMENT_DETAILS', 'PO_NUMBER', 'SELLER_ADDRESS', 'SELLER_EMAIL', 'SELLER_SITE', 'SUB_TOTAL', 'TAX', 'TITLE', 'TOTAL', 'TOTAL_WORDS', 'GSTIN_SELLER', 'OTHER', 'LOGO']
  TABLE: [[{'bbox': [[10.0, 328.88980000000004], [560.0, 220.8898]]}]]
  INVOICE_INFO: []
  BUYER: {'bbox': [[332.0, 614.4058000000001], [500.4

In [ ]:
import os, json
from pathlib import Path

ZIP_PATH = "/content/FATURA2.zip"
ANN_EXTRACT_DIR = "/content/FATURA_ANN"
ANNOT_DIR = Path(f"{ANN_EXTRACT_DIR}/invoices_dataset_final/Annotations")

# 1. Resume download with visible progress bar & extract ONLY Annotations
if not ANNOT_DIR.exists():
    !wget -c "https://zenodo.org/records/10371464/files/FATURA2.zip?download=1" -O {ZIP_PATH}
    !unzip -q -o {ZIP_PATH} "invoices_dataset_final/Annotations/*" -d {ANN_EXTRACT_DIR}

print("Annotations subfolders:", os.listdir(ANNOT_DIR))

# 2. Check OCR sample format
sample_name = "Template1_Instance0.jpg"
sample_ocr = ocr_data[sample_name]
print("\n--- OCR SAMPLE ---")
print("Keys:", list(sample_ocr.keys()), "| Size:", sample_ocr["width"], "x", sample_ocr["height"])
print("First 2 words:", sample_ocr["words"][:2])

# 3. Check Ground-Truth JSON format
json_matches = list(ANNOT_DIR.rglob("Template1_Instance0.json"))
print("\n--- FOUND JSON PATHS ---")
for p in json_matches:
    print(" ", p)

with open(json_matches[0], encoding="utf-8") as f:
    sample_ann = json.load(f)

print(f"\nTop keys ({json_matches[0].parent.name}):", list(sample_ann.keys()))
for k, v in list(sample_ann.items())[:5]:
    print(f"  {k}: {str(v)[:200]}")

Annotations subfolders: ['.virtual_documents', 'layoutlm_HF_format', 'COCO_compatible_format', 'Original_Format']

--- OCR SAMPLE ---
Keys: ['filename', 'width', 'height', 'words'] | Size: 595 x 841
First 2 words: [['00', 0.8259, 0.0514, 0.03125, 0.09833, 0.06445, 0], ['Address:16424', 0.9464, 0.04726, 0.1709, 0.19357, 0.18457, 1]]

--- FOUND JSON PATHS ---
  /content/FATURA_ANN/invoices_dataset_final/Annotations/Original_Format/Template1_Instance0.json

Top keys (Original_Format): ['TABLE', 'INVOICE_INFO', 'BUYER', 'DATE', 'DISCOUNT', 'DUE_DATE', 'GSTIN_BUYER', 'NOTE', 'PAYMENT_DETAILS', 'PO_NUMBER', 'SELLER_ADDRESS', 'SELLER_EMAIL', 'SELLER_SITE', 'SUB_TOTAL', 'TAX', 'TITLE', 'TOTAL', 'TOTAL_WORDS', 'GSTIN_SELLER', 'OTHER', 'LOGO']
  TABLE: [[{'bbox': [[10.0, 328.88980000000004], [560.0, 220.8898]]}]]
  INVOICE_INFO: []
  BUYER: {'bbox': [[332.0, 614.4058000000001], [500.43199999999996, 698.4058]], 'text': 'Bill to:Denise Perez\n16424 Timothy Mission\nMarkville, AK 58294 US\nTel:+(35

In [ ]:
from collections import Counter

ORIG_DIR = ANNOT_DIR / "Original_Format"

def pdf_box_to_norm(bbox, w, h):
    """Convert FATURA PDF bottom-left [[x1, y1], [x2, y2]] to normalized [0, 1] top-left."""
    (x1, y1), (x2, y2) = bbox[0], bbox[1]
    nx1 = min(x1, x2) / w
    nx2 = max(x1, x2) / w
    ny1 = (h - max(y1, y2)) / h
    ny2 = (h - min(y1, y2)) / h
    return [max(0.0, nx1), max(0.0, ny1), min(1.0, nx2), min(1.0, ny2)]

def word_containment(wbox, gbox):
    """Return the fraction of the word box area lying inside the ground-truth box."""
    ix1, iy1 = max(wbox[0], gbox[0]), max(wbox[1], gbox[1])
    ix2, iy2 = min(wbox[2], gbox[2]), min(wbox[3], gbox[3])
    if ix2 <= ix1 or iy2 <= iy1:
        return 0.0
    inter = (ix2 - ix1) * (iy2 - iy1)
    warea = max((wbox[2] - wbox[0]) * (wbox[3] - wbox[1]), 1e-9)
    return inter / warea

# 1. Verify coordinate alignment and word matching on a sample invoice
sample_img = "Template1_Instance0.jpg"
rec = ocr_data[sample_img]
w, h = rec["width"], rec["height"]

with open(ORIG_DIR / "Template1_Instance0.json", encoding="utf-8") as f:
    ann = json.load(f)

IGNORE_KEYS = {"OTHER", "TABLE", "LOGO"}

print("--- VERIFYING Y-FLIP & WORD MATCHING ---")
for field, val in ann.items():
    if field in IGNORE_KEYS or not isinstance(val, dict) or "bbox" not in val:
        continue
    gbox = pdf_box_to_norm(val["bbox"], w, h)
    matched = [
        wd[0] for wd in rec["words"]
        if word_containment(wd[2:6], gbox) >= 0.5
    ]
    gt_text = val.get("text", "").replace("\n", " ")[:60]
    print(f"[{field}]")
    print(f"  GT : {gt_text}")
    print(f"  OCR: {' '.join(matched)[:60]}")

# 2. Audit all entity keys and OCR image dimensions across all 10k invoices
key_counts = Counter()
sizes = Counter((r["width"], r["height"]) for r in ocr_data.values())

for jf in ORIG_DIR.glob("*.json"):
    with open(jf, encoding="utf-8") as f:
        data = json.load(f)
    for k, v in data.items():
        if isinstance(v, dict) and "bbox" in v:
            key_counts[k] += 1

print("\n--- IMAGE SIZES IN OCR ---")
print(sizes.most_common(5))

print("\n--- ENTITY COUNTS ACROSS 10K JSONS ---")
for k, c in key_counts.most_common():
    print(f"  {k:20s}: {c}")

--- VERIFYING Y-FLIP & WORD MATCHING ---
[BUYER]
  GT : Bill to:Denise Perez 16424 Timothy Mission Markville, AK 582
  OCR: Bill to:Denise Perez 16424 Timothy Mission Markville, AK 582
[DATE]
  GT : Date: 20-Mar-2008
  OCR: Date: 20-Mar-2008
[DISCOUNT]
  GT : DISCOUNT(1.85%): (-)  13.42
  OCR: DISCOUNT(1.85%6): (-) 13.42
[DUE_DATE]
  GT : Due Date : 16-Oct-2016
  OCR: Due Date : 16-Oct-2016
[GSTIN_BUYER]
  GT : GSTIN: OG@AAMFCO376K124
  OCR: GSTIN: OG@AAMFCO376K124
[NOTE]
  GT : Note: This order is shipped through blue dart courier
  OCR: Note: This order is shipped through blue dart courier
[PAYMENT_DETAILS]
  GT : Bank Name           State Bank of California Branch Name    
  OCR: Bank Name Branch Name State Bank of California Raf CAMP Bank
[PO_NUMBER]
  GT : PO Number :35
  OCR: PO Number :35
[SELLER_ADDRESS]
  GT : Address:16424 Timothy Mission Markville, AK 58294 US
  OCR: Address:16424 Timothy Mission Markville, AK 58294 US
[SELLER_EMAIL]
  GT : Email:melvin40@example.net
  OCR: 